# Legal Metrology ML: Model Training & Comparative Benchmarking

This notebook trains and benchmarks:
1. **CatBoostClassifier** (Primary Model with native categorical support)
2. **RandomForestClassifier** (Baseline Model with Ordinal Encoding)
3. **Probability Calibration** (Platt Scaling / Sigmoid for dependable risk scores)
4. **Metrology Safety Metrics** (Explicit False PASS Rate and False FAIL Rate evaluation)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.generate_dataset import generate_synthetic_dataset
from src.data.feature_engineering import engineer_features
from src.data.preprocess import MetrologyDataPreprocessor, split_metrology_dataset
from src.models.train_catboost import train_catboost_model
from src.models.train_random_forest import train_random_forest_model
from src.models.calibration import calibrate_model
from src.models.evaluate import evaluate_classifier, compare_models

## 1. Prepare Features & Partition Splits

In [ ]:
data_path = project_root / "data" / "synthetic" / "synthetic_verifications.csv"
df = pd.read_csv(data_path) if data_path.exists() else generate_synthetic_dataset(5000)
fe_df = engineer_features(df)

train_df, val_df, test_df = split_metrology_dataset(fe_df, test_size=0.15, val_size=0.15, random_state=42)

preprocessor = MetrologyDataPreprocessor()
preprocessor.fit(train_df)

X_train_cb = preprocessor.transform_catboost(train_df)
X_val_cb = preprocessor.transform_catboost(val_df)
X_test_cb = preprocessor.transform_catboost(test_df)

y_train = train_df["verification_result"]
y_val = val_df["verification_result"]
y_test = test_df["verification_result"]

X_train_rf = preprocessor.transform_rf(train_df)
X_val_rf = preprocessor.transform_rf(val_df)
X_test_rf = preprocessor.transform_rf(test_df)

## 2. Train Primary CatBoost Model

In [ ]:
cb_model, cb_meta = train_catboost_model(
    X_train_cb, y_train, X_val_cb, y_val,
    categorical_features=preprocessor.categorical_cols,
    config={"iterations": 300, "learning_rate": 0.05, "depth": 6}
)
val_cb = evaluate_classifier(cb_model, X_val_cb, y_val, model_name="CatBoost_Val")

## 3. Train Baseline Random Forest Model

In [ ]:
rf_model, rf_meta = train_random_forest_model(
    X_train_rf, y_train.values,
    feature_names=preprocessor.all_feature_cols,
    config={"n_estimators": 100, "max_depth": 10}
)
val_rf = evaluate_classifier(rf_model, X_val_rf, y_val, model_name="RandomForest_Val")

## 4. Comparative Model Selection

In [ ]:
winner, comparison = compare_models(val_cb, val_rf)
pd.DataFrame(comparison)

## 5. Test Evaluation & Explicit Metrology Safety Metrics

In [ ]:
calibrated_cb = calibrate_model(cb_model, X_val_cb, y_val, method="sigmoid")
test_metrics = evaluate_classifier(calibrated_cb, X_test_cb, y_test, model_name="Calibrated_CatBoost_Test")

print("Final Test Accuracy:   ", test_metrics["accuracy"])
print("Final Test F1 Score:   ", test_metrics["f1_score"])
print("FALSE PASS RATE:       ", test_metrics["false_pass_rate"], "<-- Critical Safety Concern")
print("FALSE FAIL RATE:       ", test_metrics["false_fail_rate"])